# 01 — Web Scraping: City Information
This notebook scrapes city data (name, country, coordinates) from Wikipedia and loads it into the `cities` table of the Gans MySQL database.

**Source:** Wikipedia  
**Target table:** `cities`  
**Run frequency:** Once (or when adding new cities)

---
## 1. Import Libraries

In [ ]:
# Uncomment to install if needed
# !conda install sqlalchemy pymysql -y

import pandas as pd
import requests
import getpass
from bs4 import BeautifulSoup

---
## 2. Database Connection
Credentials are entered at runtime using `getpass` — no secrets are stored in the notebook.

In [ ]:
password = getpass.getpass("MySQL password: ")

schema = "gans_db"
host   = "127.0.0.1"
user   = "root"
port   = 3306

connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"

---
## 3. Scraping Function
Extracts city name, country, latitude and longitude from a Wikipedia city page.

Wikipedia URLs follow a consistent pattern (`/wiki/City_Name`), and coordinates are always in a `<span class="geo-dec">` element, making them reliable to parse.

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 Chrome/134.0.0.0"}

def get_city_info(city_name):
    """Scrape city name, country, latitude and longitude from Wikipedia."""
    url = "https://en.wikipedia.org/wiki/" + city_name.replace(" ", "_")
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        print(f"[ERROR] Could not fetch {url} — status {response.status_code}")
        return None

    soup = BeautifulSoup(response.content, "html.parser")

    # Country
    try:
        country = soup.find("th", string="Country").find_next().get_text(strip=True)
    except AttributeError:
        print(f"[WARNING] Could not find country for {city_name}")
        country = "N/A"

    # Coordinates — always in <span class="geo-dec">, format: "52.5200°N 13.4050°E"
    try:
        coordinates = soup.find("span", class_="geo-dec").get_text(strip=True).split()
        latitude, longitude = None, None
        for coord in coordinates:
            value = float(coord[:-1])  # strip direction letter
            if coord[-1] in ("S", "W"):
                value = -value
            if coord[-1] in ("N", "S"):
                latitude = value
            elif coord[-1] in ("E", "W"):
                longitude = value
    except AttributeError:
        print(f"[WARNING] Could not find coordinates for {city_name}")
        latitude, longitude = None, None

    return {
        "city_name": city_name,
        "country":   country,
        "latitude":  latitude,
        "longitude": longitude
    }

---
## 4. Extract — Scrape Cities

In [ ]:
cities = ["Berlin", "Hamburg", "Munich"]

city_data = [get_city_info(city) for city in cities]
city_data = [c for c in city_data if c is not None]  # drop any failed lookups

cities_df = pd.DataFrame(city_data)
cities_df

---
## 5. Load — Push to MySQL

In [ ]:
rows_inserted = cities_df.to_sql(
    "cities",
    if_exists="append",
    con=connection_string,
    index=False
)
print(f"Inserted {rows_inserted} rows into `cities`.")